# Historical workflow (redacted, read-only)

This is an archival view of the original AWS JupyterLab notebook **cell sources**, in original order. Individual cells are rendered as Markdown rather than executable Python to prevent accidental re-running of paid SageMaker jobs, S3 writes or Pipeline updates. All execution outputs, timestamps and notebook-specific metadata were removed; account, bucket, role and personal run identifiers were redacted. This is **not** a verbatim executable notebook. Use the clean reproduction notebook or `scripts/` instead.


### Original cell 01

```python
import sys
import boto3
from importlib.metadata import version

from sagemaker.core.processing import ScriptProcessor
from sagemaker.train import ModelTrainer
from sagemaker.mlops.workflow.pipeline import Pipeline

print("=== Notebook Environment ===")
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("SageMaker SDK:", version("sagemaker"))

session = boto3.Session()
print("AWS Region:", session.region_name)

print("\n=== S3 Access Test ===")

bucket = "REDACTED_BUCKET"
s3 = session.client("s3", region_name="ap-northeast-2")

s3.head_bucket(Bucket=bucket)

print("S3 bucket access: OK")
print("SageMaker SDK imports: OK")
```

### Original cell 02

```python

import boto3
from uuid import uuid4
from sagemaker.core.helper.session_helper import get_execution_role

# 1. SageMaker 실행 역할 확인
role_arn = get_execution_role()
print("Execution role:", role_arn.rsplit("/", 1)[-1])

# 2. 프로젝트 전용 S3 버킷에 임시 객체 생성
bucket = "REDACTED_BUCKET"
key = f"_verification/{uuid4().hex}.txt"
content = b"Breast Cancer Pipeline - S3 Access Test"

s3 = boto3.client("s3", region_name="ap-northeast-2")
uploaded = False

try:
    # 쓰기 권한 검증
    s3.put_object(Bucket=bucket, Key=key, Body=content)
    uploaded = True
    print("S3 write: OK")

    # 읽기 권한 검증
    response = s3.get_object(Bucket=bucket, Key=key)
    downloaded = response["Body"].read()
    assert downloaded == content
    print("S3 read: OK")

finally:
    # 검증용 객체 삭제
    if uploaded:
        s3.delete_object(Bucket=bucket, Key=key)
        print("S3 cleanup: OK")
```